In [ ]:
import os
import random
import re
from functools import lru_cache
from typing import List

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from pypdf import PdfReader

# ----------------------------------------------------------------------------
# Setup
# ----------------------------------------------------------------------------
load_dotenv(override=True)

gemini = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.getenv("GOOGLE_API_KEY"),
)
MODEL = "gemini-flash-lite-latest"

name = "Insurance Domain Consultant"
product1 = "IC01 Principles of Insurance"

DEFAULT_PDF = r"docs\IC01 Principles of Insurance.pdf"
DEFAULT_SUMMARY = r"docs\summary.txt"

# Folder that holds all the pre-loaded study PDFs (same folder as the default PDF)
DOCS_DIR = os.path.dirname(DEFAULT_PDF)

# Subject label shown in the UI  ->  PDF file name inside DOCS_DIR
SUBJECTS = {
    "IC-01 Principles of Insurance": "IC01 Principles of Insurance.pdf",
    "IC-02 Practices of Life Insurance": "IC-02-practices-of-life-insurance.pdf",
    "IC-11 Practices of General Insurance": "IC-11-practises of general insurance.pdf",
    "IC-14 Regulations of Insurance Business": "IC-14-regulations of insurance business.pdf",
    "IC-38 IRDAI Refresher Workbook": "ic-38-irdai-refresher-workbook.pdf",
    "IC-67 Marine Insurance": "IC-67-marine-insurance.pdf",
    "IC-72 Motor Insurance": "IC-72-motor-insurance.pdf",
    "IC-74 Liability Insurance": "IC-74-liability-insurance.pdf",
    "IC-85 Reinsurance": "IC-85-reinsurance.pdf",
}
DEFAULT_SUBJECTS = ["IC-01 Principles of Insurance"]

# Size of the excerpt (in characters) sent to the model per question.
# 500 is very small and often cuts a topic mid-way; 2000-4000 gives much better questions.
MAX_CONTEXT_CHARS = 3000

RANDOM_TOPICS = [
    "health insurance basics in India",
    "term life insurance in India",
    "motor insurance (own damage, third party, IDV, NCB) in India",
    "IRDAI regulations and policyholder protection",
    "claims process: cashless vs reimbursement",
    "waiting periods, exclusions and pre-existing diseases",
    "sum insured, co-payment, deductible and sub-limits",
    "riders and add-ons",
    "underwriting and risk assessment",
    "general insurance: fire, marine and property",
    "ULIPs, endowment plans and savings-linked insurance",
    "free-look period, grace period and policy revival",
    "tax benefits on insurance (Sec 80C, 80D, 10(10D))",
    "principles of insurance: utmost good faith, insurable interest, indemnity",
    "reinsurance and risk sharing",
    "liability insurance concepts",
]

WELCOME = (
    "👋 **Welcome to the Insurance MCQ Quiz!**\n\n"
    "Before we start, choose what you'd like to practice from the panel on the left:\n\n"
    "1. 📚 **Select one or more subjects** from the pre-loaded study material\n"
    "2. 📄 *(Optional)* **Upload your own document(s)** and/or **paste your own text**\n"
    "3. 🎲 *(Optional)* **Include random insurance questions from the Web**\n"
    "4. ✍️ *(Optional)* **Type a specific area** you want to focus on (free text)\n\n"
    "💡 If you pick subject(s) and type an area but leave the random box unticked, "
    "questions stay **only within your selected subject(s)**, focused on that area where possible.\n\n"
    "If you don't pick anything, I'll use **IC-01 Principles of Insurance**.\n\n"
    "Then click **▶️ Next question** to begin!"
)


# ----------------------------------------------------------------------------
# Document loading
# ----------------------------------------------------------------------------
def read_pdf(path: str) -> str:
    reader = PdfReader(path)
    return "\n".join((p.extract_text() or "") for p in reader.pages)


def read_any(path: str) -> str:
    if path.lower().endswith(".pdf"):
        return read_pdf(path)
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()


def split_chunks(text: str, size: int = MAX_CONTEXT_CHARS, min_len: int = 200) -> List[str]:
    """Split text into chunks of ~size characters, preferring to break at paragraph/sentence ends."""
    text = re.sub(r"[ \t]+", " ", text)
    chunks, start = [], 0
    while start < len(text):
        end = min(start + size, len(text))
        if end < len(text):
            cut = max(text.rfind("\n", start, end), text.rfind(". ", start, end))
            if cut > start + size // 2:
                end = cut + 1
        chunk = text[start:end].strip()
        if len(chunk) >= min_len:  # skip tiny / empty chunks (blank pages, headers)
            chunks.append(chunk)
        start = end
    return chunks


@lru_cache(maxsize=32)
def load_chunks(path: str) -> tuple:
    """Read a file and split it into chunks (cached so PDFs are parsed only once)."""
    return tuple(split_chunks(read_any(path)))


def default_chunks() -> tuple:
    chunks = []
    if os.path.exists(DEFAULT_PDF):
        chunks += load_chunks(DEFAULT_PDF)
    if os.path.exists(DEFAULT_SUMMARY):
        chunks += load_chunks(DEFAULT_SUMMARY)
    return tuple(chunks)


def collect_documents(subjects, files, pasted_text: str = "") -> dict:
    """Return {source_name: [chunks]} for the selected subjects + uploaded files + pasted text."""
    pool = {}
    for label in subjects or []:
        path = os.path.join(DOCS_DIR, SUBJECTS[label])
        if os.path.exists(path):
            chunks = load_chunks(path)
            if chunks:
                pool[label] = list(chunks)
    for f in files or []:
        path = getattr(f, "name", f)
        chunks = load_chunks(path)
        if chunks:
            pool[os.path.basename(path)] = list(chunks)
    if pasted_text and pasted_text.strip():
        chunks = split_chunks(pasted_text.strip(), min_len=1)  # short notes are fine too
        if chunks:
            pool["Pasted text"] = chunks
    return pool


STOPWORDS = {"area", "areas", "about", "under", "topic", "topics", "related", "questions", "question", "with", "from", "that", "this", "on"}


def stem(word: str) -> str:
    """Very light stemming so 'exclusions' also matches 'exclusion'."""
    for suffix in ("ions", "ion", "ing", "es", "ed", "s"):
        if word.endswith(suffix) and len(word) - len(suffix) >= 4:
            return word[: -len(suffix)]
    return word


def keywords_from(focus: str) -> list:
    words = re.findall(r"[a-zA-Z]{4,}", focus.lower())
    return list({stem(w) for w in words if w not in STOPWORDS})


def choose_chunk(pool: dict, focus: str):
    """
    Pick a chunk ONLY from the pool (selected subjects / uploads / pasted text).
    Returns (source, chunk, matched). With a focus area, prefer chunks that mention it;
    if none do, fall back to a random chunk from the same pool (matched=False).
    """
    if focus:
        kws = keywords_from(focus)
        scored = []
        for src, chunks in pool.items():
            for c in chunks:
                low = c.lower()
                score = sum(low.count(k) for k in kws)
                if score > 0:
                    scored.append((score, src, c))
        if scored:
            scored.sort(key=lambda x: x[0], reverse=True)
            _, src, chunk = random.choice(scored[:8])  # random among best matches for variety
            return src, chunk, True

    src = random.choice(list(pool.keys()))  # pick a subject first so each gets equal weight
    return src, random.choice(pool[src]), False


# ----------------------------------------------------------------------------
# MCQ generation
# ----------------------------------------------------------------------------
class MCQ(BaseModel):
    question: str
    options: List[str]      # exactly 4 options, WITHOUT "A.", "B." prefixes
    correct_index: int      # 0-3
    explanation: str        # 1-2 lines, only shown to the user if they answer wrongly


quiz_system_prompt = f"You are {name}, a domain expert with extensive experience in the Indian insurance industry. \
Your job is to write high-quality multiple-choice questions (MCQs) for a quiz. \
Each MCQ must have exactly 4 options, exactly one clearly correct option, and plausible distractors. \
Do NOT include letters like 'A.' or 'B.' inside the options. \
The explanation must be 1-2 short lines explaining why the correct option is right. \
Never ask a question that is already in the 'already asked' list."


def generate_mcq(kind: str, content: str, focus: str, asked: list) -> MCQ:
    if kind == "document":
        user_prompt = (
            "Create ONE MCQ based strictly on the following document excerpt:\n\n"
            f"{content}\n\n"
        )
        if focus:
            user_prompt += f"The learner wants to practice this area: '{focus}'. Frame the question around it where the excerpt allows.\n\n"
    else:
        user_prompt = f"Create ONE MCQ on this Indian insurance topic: {content}.\n\n"

    if asked:
        recent = "\n".join(f"- {q}" for q in asked[-15:])
        user_prompt += f"Already asked questions (do not repeat):\n{recent}\n\n"
    user_prompt += "Vary the difficulty between easy, medium and hard."

    messages = [
        {"role": "system", "content": quiz_system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = gemini.beta.chat.completions.parse(
        model=MODEL, messages=messages, response_format=MCQ
    )
    mcq = response.choices[0].message.parsed

    # sanity check + shuffle so the correct answer isn't biased toward one position
    if len(mcq.options) != 4 or not (0 <= mcq.correct_index < 4):
        raise ValueError("Model returned a malformed MCQ")
    correct_text = mcq.options[mcq.correct_index]
    random.shuffle(mcq.options)
    mcq.correct_index = mcq.options.index(correct_text)
    return mcq


# ----------------------------------------------------------------------------
# Choosing where the next question comes from
# ----------------------------------------------------------------------------
def pick_source(subjects, files, pasted_text, include_random: bool, focus: str):
    """Returns (source_label, kind, content, note). kind is 'document' or 'topic'."""
    pool = collect_documents(subjects, files, pasted_text)

    # nothing selected at all -> default document (IC01 + summary)
    if not pool and not include_random:
        chunks = default_chunks()
        if chunks:
            pool = {product1: list(chunks)}
        else:
            include_random = True  # default files missing, so fall back to random

    options = []
    if pool:
        options.append("docs")
    if include_random:
        options.append("random")

    if random.choice(options) == "docs":
        src, chunk, matched = choose_chunk(pool, focus)
        note = ""
        if focus and not matched:
            note = f" (nothing on '{focus}' found here, so this is a general question from this subject)"
        return src, "document", chunk, note

    topic = focus or random.choice(RANDOM_TOPICS)
    return "Random (Web / general knowledge)", "topic", topic, ""


# ----------------------------------------------------------------------------
# Gradio handlers
# ----------------------------------------------------------------------------
LETTERS = ["A", "B", "C", "D"]


def new_state():
    return {"mcq": None, "answered": True, "score": 0, "total": 0, "asked": []}


def welcome_chat():
    return [{"role": "assistant", "content": WELCOME}]


def score_text(state):
    return f"**Score:** {state['score']} / {state['total']}"


def next_question(chat, state, subjects, files, pasted_text, include_random, focus):
    chat = chat or []
    state = state or new_state()
    focus = (focus or "").strip()

    try:
        source, kind, content, note = pick_source(subjects, files, pasted_text, include_random, focus)
        mcq = generate_mcq(kind, content, focus, state["asked"])
    except Exception as e:
        chat.append({"role": "assistant", "content": f"⚠️ Couldn't generate a question: {e}\n\nPlease try again."})
        return [chat, state, *[gr.update() for _ in range(4)], score_text(state)]

    state["mcq"] = mcq.model_dump()
    state["answered"] = False
    state["asked"].append(mcq.question)

    body = f"**Q{len(state['asked'])}. {mcq.question}**\n\n"
    body += "\n".join(f"**{LETTERS[i]}.** {opt}" for i, opt in enumerate(mcq.options))
    body += f"\n\n<sub>📖 Source: {source}{note}</sub>"
    chat.append({"role": "assistant", "content": body})

    buttons = [
        gr.update(value=f"{LETTERS[i]}. {opt}", interactive=True)
        for i, opt in enumerate(mcq.options)
    ]
    return [chat, state, *buttons, score_text(state)]


def make_answer_handler(idx: int):
    def handler(chat, state):
        chat = chat or []
        if not state or not state.get("mcq") or state["answered"]:
            return [chat, state, *[gr.update() for _ in range(4)], score_text(state or new_state())]

        mcq = state["mcq"]
        state["answered"] = True
        state["total"] += 1

        chat.append({"role": "user", "content": f"{LETTERS[idx]}. {mcq['options'][idx]}"})

        if idx == mcq["correct_index"]:
            state["score"] += 1
            reply = "✅ **Right!**"
        else:
            right = mcq["correct_index"]
            reply = (
                f"❌ **Wrong.** The correct answer is **{LETTERS[right]}. {mcq['options'][right]}**.\n\n"
                f"{mcq['explanation']}"
            )  # explanation shown ONLY when the answer is wrong
        chat.append({"role": "assistant", "content": reply})

        locked = [gr.update(interactive=False) for _ in range(4)]
        return [chat, state, *locked, score_text(state)]

    return handler


def reset_quiz():
    state = new_state()
    return [welcome_chat(), state, *[gr.update(value=l, interactive=False) for l in LETTERS], score_text(state)]


# ----------------------------------------------------------------------------
# UI
# ----------------------------------------------------------------------------
with gr.Blocks(title="Insurance MCQ Quiz") as demo:
    gr.Markdown(f"# 🛡️ {name} - MCQ Quiz")

    state = gr.State(new_state())

    with gr.Row():
        with gr.Column(scale=1):
            subjects = gr.CheckboxGroup(
                choices=list(SUBJECTS.keys()),
                value=DEFAULT_SUBJECTS,
                label="📚 Select subject(s)",
            )
            files = gr.File(
                label="📄 Upload your own document(s) (PDF / TXT / MD)",
                file_count="multiple",
                file_types=[".pdf", ".txt", ".md"],
            )
            pasted_text = gr.Textbox(
                label="📝 Or paste your own text / notes (optional)",
                placeholder="Paste any insurance text here to be quizzed on it...",
                lines=4,
            )
            include_random = gr.Checkbox(
                value=False,
                label="🎲 Include random insurance questions from the Web",
            )
            focus = gr.Textbox(
                label="✍️ Specific area to practice (optional)",
                placeholder="e.g. utmost good faith, third-party liability, IRDAI grievance redressal...",
                lines=2,
            )
            score_md = gr.Markdown("**Score:** 0 / 0")
            next_btn = gr.Button("▶️ Next question", variant="primary")
            reset_btn = gr.Button("🔄 Reset quiz")

        with gr.Column(scale=2):
            chatbot = gr.Chatbot(value=welcome_chat(), height=480, label="Quiz")
            answer_btns = [
                gr.Button(l, interactive=False, variant="secondary") for l in LETTERS
            ]

    outputs = [chatbot, state, *answer_btns, score_md]

    next_btn.click(
        next_question,
        inputs=[chatbot, state, subjects, files, pasted_text, include_random, focus],
        outputs=outputs,
    )
    reset_btn.click(reset_quiz, inputs=None, outputs=outputs)
    for i, btn in enumerate(answer_btns):
        btn.click(make_answer_handler(i), inputs=[chatbot, state], outputs=outputs)

if __name__ == "__main__":
    demo.launch()